In [35]:
import pandas as pd

df = pd.read_csv("D:/Hand_Gesture/data/hagrid_set_v1_medium_processed/labels.csv")

print(df.head())
print("total:", len(df))
print(df["label_name"].value_counts())
print(df["label"].value_counts().sort_index())

                                         idx original_class_folder  label  \
0  call_007302a2-fdee-4165-977e-9f1374b8d547                  0_NA      0   
1  call_05eef5c2-d9ad-4aa2-a1f4-bd74cecde11f                  0_NA      0   
2  call_07f7b373-914a-47f2-b83c-8658dce4369a                  0_NA      0   
3  call_0990c11e-558a-458c-b285-75a641af66d0                  0_NA      0   
4  call_09c8587e-44bf-41d5-94c6-9f1db2baa8f9                  0_NA      0   

  label_name                                          crop_path  \
0        N_A  D:\Hand_Gesture\data\hagrid_set_v1_medium_proc...   
1        N_A  D:\Hand_Gesture\data\hagrid_set_v1_medium_proc...   
2        N_A  D:\Hand_Gesture\data\hagrid_set_v1_medium_proc...   
3        N_A  D:\Hand_Gesture\data\hagrid_set_v1_medium_proc...   
4        N_A  D:\Hand_Gesture\data\hagrid_set_v1_medium_proc...   

                                       landmark_path quality  
0  D:\Hand_Gesture\data\hagrid_set_v1_medium_proc...      ok  
1  D:\Ha

# Split: Training / Validation / Testing

In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

df = pd.read_csv("D:/Hand_Gesture/data/hagrid_set_v1_medium_processed/labels.csv")
# 如果你還在用 sample，就改成：
# df = pd.read_csv("D:/Hand_Gesture/data/processed_sample/labels.csv")

# 先切出 train 70%，temp 30%
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

# 再把 temp 30% 切成 val 15%、test 15%
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

print("\nTrain class count:")
print(train_df["label_name"].value_counts())

print("\nVal class count:")
print(val_df["label_name"].value_counts())

print("\nTest class count:")
print(test_df["label_name"].value_counts())

train: 6991
val: 1498
test: 1499

Train class count:
label_name
N_A     3491
like     700
fist     700
ok       700
palm     700
one      700
Name: count, dtype: int64

Val class count:
label_name
N_A     748
palm    150
ok      150
like    150
fist    150
one     150
Name: count, dtype: int64

Test class count:
label_name
N_A     749
like    150
palm    150
one     150
ok      150
fist    150
Name: count, dtype: int64


In [37]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
model = mobilenet_v3_small(weights=weights)

# MobileNetV3-small 原本是 ImageNet 1000 類
# 改成作業需要的 6 類
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 6)

model = model.to(device)

device: cuda


In [38]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)
print("device count:", torch.cuda.device_count())

torch version: 2.8.0+cu129
cuda available: True
torch cuda version: 12.9
device count: 1


In [39]:
import sys
!{sys.executable} -m pip install torch torchvision
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_transform = base_transform
val_transform = base_transform
test_transform = base_transform

# Image only model

In [ ]:
class GestureImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["crop_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform is not None:
            img = self.transform(img)

        return img, label


train_dataset = GestureImageDataset(train_df, transform=train_transform)
val_dataset = GestureImageDataset(val_df, transform=val_transform)
test_dataset = GestureImageDataset(val_df, transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

imgs, labels = next(iter(train_loader))
print(imgs.shape)
print(labels)

torch.Size([16, 3, 224, 224])
tensor([0, 0, 1, 5, 1, 3, 4, 4, 4, 1, 1, 3, 3, 2, 3, 0])


In [6]:
import sys
!{sys.executable} -m pip install -U tqdm ipywidgets
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from pathlib import Path

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

save_path = Path("D:/Hand_Gesture/model/mobilenetv3_small_best.pth")
save_path.parent.mkdir(exist_ok=True)

def train_one_epoch(model, loader):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for imgs, labels in pbar:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(imgs)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        current_loss = total_loss / total
        current_acc = correct / total

        pbar.set_postfix({
            "loss": f"{current_loss:.4f}",
            "acc": f"{current_acc:.4f}"
        })

    return total_loss / total, correct / total


def evaluate(model, loader):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for imgs, labels in pbar:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            loss = criterion(logits, labels)

            total_loss += loss.item() * imgs.size(0)

            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            current_loss = total_loss / total
            current_acc = correct / total

            pbar.set_postfix({
                "loss": f"{current_loss:.4f}",
                "acc": f"{current_acc:.4f}"
            })

    return total_loss / total, correct / total


best_val_acc = 0.0
num_epochs = 15

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model: val_acc={best_val_acc:.4f}")

print("\nBest val acc:", best_val_acc)
print("Best model saved to:", save_path)


Epoch 1/15


Training:   0%|          | 0/461 [00:00<?, ?it/s]

Validation:   0%|          | 0/116 [00:00<?, ?it/s]

train_loss=0.7027, train_acc=0.7452, val_loss=0.3133, val_acc=0.9011
Saved best model: val_acc=0.9011

Epoch 2/15


Training:   0%|          | 0/461 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/116 [00:00<?, ?it/s]

train_loss=0.0416, train_acc=0.9862, val_loss=0.2895, val_acc=0.9316

Epoch 15/15


Training:   0%|          | 0/461 [00:00<?, ?it/s]

Validation:   0%|          | 0/116 [00:00<?, ?it/s]

train_loss=0.0524, train_acc=0.9825, val_loss=0.3072, val_acc=0.9321

Best val acc: 0.9429657794676806
Best model saved to: D:\Hand_Gesture\model\mobilenetv3_small_best.pth


In [ ]:
from pathlib import Path

model_dir = Path("D:/Hand_Gesture/model")
model_dir.mkdir(exist_ok=True)

torch.save(model.state_dict(), model_dir / "mobilenetv3_small_best.pth")

print("saved to:", model_dir / "mobilenetv3_small_best.pth")

saved to: D:\Hand_Gesture\model\mobilenetv3_small_best.pth


In [8]:
from pathlib import Path

model_path = Path("D:/Hand_Gesture/model/mobilenetv3_small_best.pth")

file_size_mb = model_path.stat().st_size / (1024 ** 2)

print("Model file:", model_path)
print("Saved .pth size:", f"{file_size_mb:.2f} MB")

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

param_size_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_size_bytes = sum(b.numel() * b.element_size() for b in model.buffers())

model_memory_mb = (param_size_bytes + buffer_size_bytes) / (1024 ** 2)

print("Total parameters:", f"{num_params / 1e6:.3f} M")
print("Trainable parameters:", f"{trainable_params / 1e6:.3f} M")
print("Parameter + buffer size:", f"{model_memory_mb:.2f} MB")

Model file: D:\Hand_Gesture\model\mobilenetv3_small_best.pth
Saved .pth size: 5.94 MB
Total parameters: 1.524 M
Trainable parameters: 1.524 M
Parameter + buffer size: 5.86 MB


# 1.a : output feature vectors 

In [40]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


class MobileNetV3SmallFeatureExtractor(nn.Module):
    def __init__(self, output_dim=128):
        super().__init__()

        weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
        backbone = mobilenet_v3_small(weights=weights)

        # 保留 MobileNetV3-small 的 CNN feature extractor
        self.features = backbone.features
        self.avgpool = backbone.avgpool

        # MobileNetV3-small avgpool 後通常是 576 維
        self.projector = nn.Sequential(
            nn.Linear(576, output_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.projector(x)
        return x

In [41]:
image_encoder = MobileNetV3SmallFeatureExtractor(output_dim=128)
image_encoder = image_encoder.to(device)

# image + landmark model

In [42]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


class MobileNetV3SmallFeatureExtractor(nn.Module):
    def __init__(self, output_dim=128):
        super().__init__()

        weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
        backbone = mobilenet_v3_small(weights=weights)

        self.features = backbone.features
        self.avgpool = backbone.avgpool

        self.projector = nn.Sequential(
            nn.Linear(576, output_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.projector(x)
        return x


class LandmarkMLP(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=64, output_dim=128):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, landmarks):
        # landmarks: [B, 21, 2] 或 [B, 42]
        if landmarks.dim() == 3:
            landmarks = landmarks.view(landmarks.size(0), -1)

        return self.mlp(landmarks)


class GestureFusionModel(nn.Module):
    def __init__(self, image_dim=128, landmark_dim=128, num_classes=6):
        super().__init__()

        self.image_encoder = MobileNetV3SmallFeatureExtractor(output_dim=image_dim)
        self.landmark_encoder = LandmarkMLP(output_dim=landmark_dim)

        fusion_dim = image_dim + landmark_dim  # 128 + 128 = 256

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, img, landmarks):
        image_feature = self.image_encoder(img)          # [B, 128]
        landmark_feature = self.landmark_encoder(landmarks)  # [B, 128]

        fusion_feature = torch.cat(
            [image_feature, landmark_feature],
            dim=1
        )  # [B, 256]

        logits = self.classifier(fusion_feature)  # [B, 6]

        return logits

In [43]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np


class GestureFusionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["crop_path"]).convert("RGB")
        landmarks = np.load(row["landmark_path"]).astype(np.float32)
        label = int(row["label"])

        if self.transform is not None:
            img = self.transform(img)

        landmarks = torch.tensor(landmarks, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)

        return img, landmarks, label

In [44]:
train_dataset = GestureFusionDataset(train_df, transform=train_transform)
val_dataset = GestureFusionDataset(val_df, transform=val_transform)
test_dataset = GestureFusionDataset(test_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [45]:
model = GestureFusionModel(
    image_dim=128,
    landmark_dim=128,
    num_classes=6
).to(device)

print(model)

GestureFusionModel(
  (image_encoder): MobileNetV3SmallFeatureExtractor(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
            (activation): ReLU()
            (scale_activation): Hardsigmoid

In [46]:
# check shapes

imgs, landmarks, labels = next(iter(train_loader))

imgs = imgs.to(device)
landmarks = landmarks.to(device)

with torch.no_grad():
    logits = model(imgs, landmarks)

print("imgs:", imgs.shape)
print("landmarks:", landmarks.shape)
print("labels:", labels.shape)
print("logits:", logits.shape)

imgs: torch.Size([32, 3, 224, 224])
landmarks: torch.Size([32, 21, 2])
labels: torch.Size([32])
logits: torch.Size([32, 6])


In [47]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for imgs, landmarks, labels in tqdm(loader, desc="Training", leave=False):
        imgs = imgs.to(device)
        landmarks = landmarks.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(imgs, landmarks)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, landmarks, labels in tqdm(loader, desc="Validation", leave=False):
            imgs = imgs.to(device)
            landmarks = landmarks.to(device)
            labels = labels.to(device)

            logits = model(imgs, landmarks)
            loss = criterion(logits, labels)

            total_loss += loss.item() * imgs.size(0)

            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [48]:
import torch.optim as optim
from pathlib import Path
from tqdm.notebook import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

save_path = Path("D:/Hand_Gesture/model/fusion_mobilenetv3_landmark_set_v1_medium_best.pth") # !!!!!!
save_path.parent.mkdir(exist_ok=True)

best_val_acc = 0.0
num_epochs = 15

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model: val_acc={best_val_acc:.4f}")

print("\nBest validation accuracy:", best_val_acc)
print("Best model saved to:", save_path)


Epoch 1/15


Training:   0%|          | 0/219 [00:00<?, ?it/s]

Validation:   0%|          | 0/47 [00:00<?, ?it/s]

train_loss=1.1484, train_acc=0.5906, val_loss=0.5806, val_acc=0.8571
Saved best model: val_acc=0.8571

Epoch 2/15


Training:   0%|          | 0/219 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/47 [00:00<?, ?it/s]

train_loss=0.0573, train_acc=0.9814, val_loss=0.3870, val_acc=0.9212

Epoch 15/15


Training:   0%|          | 0/219 [00:00<?, ?it/s]

Validation:   0%|          | 0/47 [00:00<?, ?it/s]

train_loss=0.0427, train_acc=0.9861, val_loss=0.3883, val_acc=0.9166

Best validation accuracy: 0.9265687583444593
Best model saved to: D:\Hand_Gesture\model\fusion_mobilenetv3_landmark_set_v1_medium_best.pth


In [52]:
import torch
import pandas as pd
import numpy as np

class_names = ["N_A", "fist", "like", "ok", "one", "palm"]

def evaluate_detailed(model, loader, criterion=None, threshold=None):
    model.eval()

    total_loss = 0.0
    total = 0

    all_labels = []
    all_preds = []
    all_confs = []

    with torch.no_grad():
        for imgs, landmarks, labels in loader:
            imgs = imgs.to(device)
            landmarks = landmarks.to(device)
            labels = labels.to(device)

            logits = model(imgs, landmarks)

            if criterion is not None:
                loss = criterion(logits, labels)
                total_loss += loss.item() * imgs.size(0)

            probs = torch.softmax(logits, dim=1)
            confs, preds = torch.max(probs, dim=1)

            # 如果有設 threshold，低於 threshold 的預測改成 N_A
            if threshold is not None:
                preds = torch.where(
                    confs < threshold,
                    torch.zeros_like(preds),
                    preds
                )

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_confs.extend(confs.cpu().numpy())

            total += labels.size(0)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_confs = np.array(all_confs)

    accuracy = (all_labels == all_preds).mean()

    target_mask = all_labels != 0
    na_mask = all_labels == 0

    target_accuracy = (
        (all_labels[target_mask] == all_preds[target_mask]).mean()
        if target_mask.sum() > 0 else 0
    )

    na_false_trigger_rate = (
        (all_preds[na_mask] != 0).mean()
        if na_mask.sum() > 0 else 0
    )

    avg_loss = total_loss / total if criterion is not None else None

    metrics = {
        "loss": avg_loss,
        "accuracy": accuracy,
        "target_accuracy": target_accuracy,
        "na_false_trigger_rate": na_false_trigger_rate,
    }

    result_df = pd.DataFrame({
        "true": all_labels,
        "pred": all_preds,
        "confidence": all_confs,
        "true_name": [class_names[i] for i in all_labels],
        "pred_name": [class_names[i] for i in all_preds],
    })

    cm = pd.crosstab(
        result_df["true_name"],
        result_df["pred_name"],
        rownames=["True"],
        colnames=["Pred"],
        dropna=False
    )

    # 補齊沒有出現的 row / column，讓表格固定 6 類
    cm = cm.reindex(index=class_names, columns=class_names, fill_value=0)

    return metrics, cm, result_df

In [53]:
metrics, cm, result_df = evaluate_detailed(
    model,
    test_loader,
    criterion=criterion,
    threshold=None
)

print(metrics)
display(cm)
display(result_df.head())

{'loss': 0.2709164931038127, 'accuracy': np.float64(0.9472981987991995), 'target_accuracy': np.float64(0.9333333333333333), 'na_false_trigger_rate': np.float64(0.03871829105473965)}


Pred,N_A,fist,like,ok,one,palm
True,,,,,,
N_A,720,6,10,2,7,4
fist,8,139,0,0,3,0
like,4,0,144,1,1,0
ok,11,1,1,136,0,1
one,11,1,1,0,136,1
palm,3,0,0,2,0,145


,true,pred,confidence,true_name,pred_name
0,0,0,0.999980,N_A,N_A
1,0,0,1.000000,N_A,N_A
2,0,0,0.998968,N_A,N_A
3,0,0,1.000000,N_A,N_A
4,2,2,0.999883,like,like


In [51]:
from pathlib import Path

file_size_mb = save_path.stat().st_size / (1024 ** 2)

print("Model file:", save_path)
print("Saved .pth size:", f"{file_size_mb:.2f} MB")

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

param_size_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_size_bytes = sum(b.numel() * b.element_size() for b in model.buffers())

model_memory_mb = (param_size_bytes + buffer_size_bytes) / (1024 ** 2)

print("Total parameters:", f"{num_params / 1e6:.3f} M")
print("Trainable parameters:", f"{trainable_params / 1e6:.3f} M")
print("Parameter + buffer size:", f"{model_memory_mb:.2f} MB")

Model file: D:\Hand_Gesture\model\fusion_mobilenetv3_landmark_set_v1_medium_best.pth
Saved .pth size: 4.16 MB
Total parameters: 1.050 M
Trainable parameters: 1.050 M
Parameter + buffer size: 4.05 MB


# Results

## 1. image only 
train_loss=0.0524, train_acc=0.9825, val_loss=0.3072, val_acc=0.9321

Best val acc: 0.9429657794676806

Best model saved to: D:\Hand_Gesture\model\mobilenetv3_small_best.pth

## 2. image + validation
- hidden dim = 64:
  
    {'loss': 0.29086411815249885, 'accuracy': np.float64(0.9319826338639653), 'target_accuracy': np.float64(0.8973607038123167), 'na_false_trigger_rate': np.float64(0.03428571428571429)}

    {'loss': 0.2709164931038127, 'accuracy': np.float64(0.9472981987991995), 'target_accuracy': np.float64(0.9333333333333333), 'na_false_trigger_rate': np.float64(0.03871829105473965)}

    testing accuracy: 0.8973607038123167 / 0.9333333333333333
    
    Best model saved to: D:\Hand_Gesture\model\fusion_mobilenetv3_landmark_best.pth